<details>
<summary> 팀 공유 개선본v5.1 로그</summary>

이 노트북은 `00.Project_Fertility_PSP_v5.ipynb`와 비슷한 보고서 흐름을 따르도록 다시 정리한 팀 공유용 독립본입니다.

중요한 분리 원칙은 다음과 같습니다.

- 기존에 별도로 진행 중인 우리 연구 라인, stage 실험, public score anchor, 이전 제출 후보는 사용하지 않습니다.
- 입력은 현재 프로젝트 폴더의 `train.csv`, `test.csv`, `sample_submission.csv`만 사용합니다.
- 출력은 `outputs/team_report_v6_1/` 아래에만 저장합니다.
- v5의 장점인 긴 설명형 구조는 살리되, 후반부 실행 순서 의존 오류가 생기지 않도록 예측값은 `prediction_bank`로 통합 관리합니다.

</details>

## 00. 프로젝트 개요 및 분석 전략

목표는 난임 환자별 임신 성공 여부를 예측하는 이진 분류 모델을 만드는 것입니다.

이 보고서의 전략은 네 단계입니다.

1. EDA에서 성공률 차이와 구조적 결측을 확인합니다.
2. 결측, 나이, 시술 유형, 배아/난자 정보를 모델이 이해하기 쉬운 feature로 바꿉니다.
3. CatBoost, LightGBM, XGBoost를 5-Fold OOF 기준으로 비교합니다.
4. OOF ROC-AUC를 기준으로 raw average, rank average, hybrid ensemble 후보를 생성합니다.

ROC-AUC는 확률의 절대값보다 성공 가능성이 높은 환자를 더 위에 잘 올렸는지를 보는 지표입니다. 따라서 최종 비교는 OOF ROC-AUC 중심으로 진행하되, LogLoss도 함께 확인합니다.


## 목차

1. 데이터 탐색(EDA)  
   1-1. 데이터 로드 및 기본 구조 확인  
   1-2. 타깃 분포 및 클래스 불균형 확인  
   1-3. 결측치 현황 확인  
   1-4. 시술 유형별 구조적 결측 확인  
   1-5. 주요 변수별 임신 성공률 확인  
   1-6. 나이·난자·배아 구간 탐색  
   1-7. 이상치 후보 확인  
   1-8. 상관관계 확인  
   1-9. EDA 종합 정리  

2. 데이터 전처리 및 Feature Engineering  
   2-1. 학습용 데이터 구성  
   2-2. 결측 flag 생성  
   2-3. 이상치 clipping  
   2-4. 임상 지식 기반 파생변수  
   2-5. 구간화 변수  
   2-6. 조합 변수 및 비율 변수  
   2-7. 범주형/수치형 변수 정리  
   2-8. 인코딩 전략  
   2-9. Fold 단위 전처리  
   2-10. K-Fold 설정  

3. 모델링  
   3-1. Baseline 모델 성능 비교  
   3-2. CatBoost 5-Fold OOF  
   3-3. LightGBM 5-Fold OOF  
   3-4. XGBoost 5-Fold OOF  
   3-5. 모델 성능 비교 시각화  
   3-6. Feature Importance 확인  
   3-7. Ensemble 후보 비교  
   3-8. 세밀 가중치 탐색  
   3-9. Rank Average 앙상블  
   3-10. Raw + Rank Hybrid 앙상블  

4. 평가  
5. 제출 및 리더보드 확인


## 01. 데이터 탐색(EDA)

EDA는 모든 변수를 깊게 파고들기보다, 이후 feature engineering으로 연결되는 신호를 찾는 데 집중합니다.

특히 이 데이터에서는 결측치가 단순 누락이 아닐 가능성이 큽니다. IVF와 DI는 실제 시술 과정이 다르기 때문에 배아·난자·이식 관련 값이 비어 있는 방식도 달라질 수 있습니다.


### 01-1. 데이터 로드 및 기본 구조 확인


In [ ]:
# 라이브러리 및 실행 환경 세팅

from __future__ import annotations

import importlib.util
import json
import os
import random
import re
import warnings
from dataclasses import dataclass, field
from itertools import combinations
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import matplotlib.pyplot as plt
except Exception as e:
    plt = None
    print('matplotlib import skipped:', e)

try:
    from catboost import CatBoostClassifier, Pool
except Exception as e:
    CatBoostClassifier = None
    Pool = None
    print('CatBoost import skipped:', e)

try:
    import lightgbm as lgb
except Exception as e:
    lgb = None
    print('LightGBM import skipped:', e)

try:
    from xgboost import XGBClassifier
except Exception as e:
    XGBClassifier = None
    print('XGBoost import skipped:', e)

required_packages = ['numpy', 'pandas', 'scipy', 'sklearn']
missing_required = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]
if missing_required:
    raise RuntimeError('필수 패키지가 없습니다: ' + ', '.join(missing_required))

print('필수 패키지 확인 완료')


In [ ]:
# 실행 설정

@dataclass
class Config:
    run_mode: str = 'quick'  # quick: 동작 확인 / full: 제출 후보 생성
    n_splits: int = 5
    target: str = '임신 성공 여부'
    id_col: str = 'ID'
    missing_token: str = '__MISSING__'
    random_state: int = 42
    use_catboost_gpu: bool = False
    data_dir: Path = field(default_factory=lambda: Path.cwd())
    out_dir: Path = field(default_factory=lambda: Path.cwd() / 'outputs' / 'team_report_v6_1')

    def seeds(self) -> List[int]:
        return [42] if self.run_mode == 'quick' else [42, 2026, 777]

    def catboost_iterations(self) -> int:
        return 80 if self.run_mode == 'quick' else 1600

    def lightgbm_estimators(self) -> int:
        return 120 if self.run_mode == 'quick' else 2400

    def xgboost_estimators(self) -> int:
        return 100 if self.run_mode == 'quick' else 1800

cfg = Config(run_mode='quick')
cfg.out_dir.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(cfg.random_state)

print('RUN_MODE:', cfg.run_mode)
print('N_SPLITS:', cfg.n_splits)
print('SEEDS:', cfg.seeds())
print('OUT_DIR:', cfg.out_dir)


In [ ]:
# 데이터 로드

REQUIRED_FILES = ['train.csv', 'test.csv', 'sample_submission.csv']


def find_data_dir(candidates: Iterable[Path]) -> Path:
    for path in candidates:
        path = Path(path)
        if all((path / name).exists() for name in REQUIRED_FILES):
            return path
    checked = '\n'.join(str(Path(p).resolve()) for p in candidates)
    raise FileNotFoundError(
        'train.csv, test.csv, sample_submission.csv를 찾지 못했습니다.\n'
        '노트북을 데이터 파일이 있는 폴더에서 실행하거나, ./data 폴더에 파일을 넣어주세요.\n\n'
        '확인한 경로:\n' + checked
    )

cfg.data_dir = find_data_dir([
    Path.cwd(),
    Path.cwd() / 'data',
    Path.cwd().parent / 'data',
    Path('/mnt/data'),
])

train = pd.read_csv(cfg.data_dir / 'train.csv')
test = pd.read_csv(cfg.data_dir / 'test.csv')
sample_submission = pd.read_csv(cfg.data_dir / 'sample_submission.csv')

print('DATA_DIR:', cfg.data_dir)
print('train:', train.shape)
print('test:', test.shape)
print('sample_submission:', sample_submission.shape)
display(train.head())


In [ ]:
# train/test 컬럼 구조 확인 및 입력 검증


def validate_inputs(train: pd.DataFrame, test: pd.DataFrame, sample_submission: pd.DataFrame, cfg: Config) -> None:
    if cfg.target not in train.columns:
        raise ValueError(f'train에 target 컬럼이 없습니다: {cfg.target}')
    if cfg.target in test.columns:
        raise ValueError(f'test에 target 컬럼이 들어 있습니다. 누수 위험: {cfg.target}')
    if cfg.id_col not in train.columns or cfg.id_col not in test.columns:
        raise ValueError(f'train/test에 ID 컬럼이 필요합니다: {cfg.id_col}')
    if cfg.id_col not in sample_submission.columns:
        raise ValueError(f'sample_submission에 ID 컬럼이 필요합니다: {cfg.id_col}')
    if len(test) != len(sample_submission):
        raise ValueError('test와 sample_submission의 행 수가 다릅니다.')
    if not (test[cfg.id_col].astype(str).values == sample_submission[cfg.id_col].astype(str).values).all():
        raise ValueError('test와 sample_submission의 ID 순서가 다릅니다.')
    if train[cfg.target].isna().any():
        raise ValueError('target에 결측값이 있습니다.')

validate_inputs(train, test, sample_submission, cfg)

common_cols = sorted(set(train.columns) & set(test.columns))
train_only = sorted(set(train.columns) - set(test.columns))
test_only = sorted(set(test.columns) - set(train.columns))

print('common columns:', len(common_cols))
print('train only:', train_only)
print('test only:', test_only)
print('input validation completed')


#### 01-1. 결과 해석

train에는 target인 `임신 성공 여부`가 있어야 하고, test에는 없어야 합니다.  
또한 sample submission의 ID 순서가 test와 같아야 제출 파일을 안전하게 만들 수 있습니다.


### 01-2. 타깃 분포 및 클래스 불균형 확인


In [ ]:
# 타깃 분포 확인

target_counts = train[cfg.target].value_counts(dropna=False).sort_index().to_frame('count')
target_counts['ratio'] = target_counts['count'] / len(train)
display(target_counts)
print('target mean:', float(train[cfg.target].mean()))

if plt is not None:
    ax = target_counts['count'].plot(kind='bar', title='Target distribution')
    ax.set_xlabel(cfg.target)
    ax.set_ylabel('count')
    plt.show()


#### 01-2. 결과 해석

성공/실패 비율이 균형적이지 않으면 정확도만으로 모델을 평가하기 어렵습니다.  
이 보고서에서는 ROC-AUC를 중심 지표로 두고, LogLoss는 확률 보정 상태를 확인하는 보조 지표로 사용합니다.


### 01-3. 결측치 현황 확인


In [ ]:
# 결측치 현황 확인

missing_summary = (
    train.drop(columns=[cfg.target], errors='ignore')
    .isna()
    .agg(['sum', 'mean'])
    .T
    .rename(columns={'sum': 'missing_count', 'mean': 'missing_rate'})
    .sort_values('missing_rate', ascending=False)
)
missing_summary['missing_count'] = missing_summary['missing_count'].astype(int)
display(missing_summary.head(30))

high_missing_cols = missing_summary.query('missing_rate >= 0.3').index.tolist()
print('30% 이상 결측 컬럼 수:', len(high_missing_cols))
print(high_missing_cols[:30])


#### 01-3. 결과 해석

결측률이 높다고 바로 제거하지 않습니다. 난임 시술 데이터에서는 특정 과정이 실제로 진행되지 않았기 때문에 값이 비어 있을 수 있습니다.  
따라서 결측 여부 자체를 feature로 남기는 것이 중요합니다.


### 01-4. 시술 유형별 구조적 결측 확인


In [ ]:
# 구조적 결측 후보 컬럼 정의

EMBRYO_COLS = [
    '총 생성 배아 수',
    '미세주입된 난자 수',
    '미세주입에서 생성된 배아 수',
    '이식된 배아 수',
    '미세주입 배아 이식 수',
    '저장된 배아 수',
    '미세주입 후 저장된 배아 수',
    '해동된 배아 수',
    '해동 난자 수',
    '수집된 신선 난자 수',
    '저장된 신선 난자 수',
    '혼합된 난자 수',
    '파트너 정자와 혼합된 난자 수',
    '기증자 정자와 혼합된 난자 수',
]

present_embryo_cols = [col for col in EMBRYO_COLS if col in train.columns]
print('present embryo/egg columns:', len(present_embryo_cols))
print(present_embryo_cols)


In [ ]:
# 시술 유형별 결측 패턴 확인

if present_embryo_cols:
    tmp = train[[cfg.target] + present_embryo_cols].copy()
    if '시술 유형' in train.columns:
        tmp['시술 유형'] = train['시술 유형']
    tmp['embryo_missing_count'] = tmp[present_embryo_cols].isna().sum(axis=1)
    tmp['embryo_all_missing'] = (tmp['embryo_missing_count'] == len(present_embryo_cols)).astype('int8')
    display(
        tmp.groupby('embryo_all_missing')[cfg.target]
        .agg(count='count', success_rate='mean')
        .rename(index={0: 'not_all_missing', 1: 'all_missing'})
    )
    if '시술 유형' in tmp.columns:
        display(
            tmp.groupby(['시술 유형', 'embryo_all_missing'])[cfg.target]
            .agg(count='count', success_rate='mean')
            .sort_values('count', ascending=False)
            .head(30)
        )
else:
    print('배아/난자 관련 후보 컬럼이 없습니다.')


#### 01-4. 결과 해석

배아·난자 관련 값이 한꺼번에 비어 있는 행은 단순 누락일 수도 있지만, 시술 과정 차이에서 생긴 구조적 결측일 수 있습니다.  
이 정보는 이후 `f_embryo_missing_count`, `f_embryo_all_missing`, 개별 missing flag로 반영합니다.


### 01-5. 주요 변수별 임신 성공률 확인


In [ ]:
# 주요 범주형 변수별 성공률 확인 함수


def display_group_rate(df: pd.DataFrame, col: str, target: str, min_count: int = 30, top_n: int = 30) -> None:
    if col not in df.columns:
        print(f'컬럼 없음: {col}')
        return
    tab = (
        df.groupby(col, dropna=False)[target]
        .agg(count='count', success_rate='mean')
        .sort_values(['count', 'success_rate'], ascending=[False, False])
    )
    tab = tab[tab['count'] >= min_count]
    display(tab.head(top_n))

for col in ['시술 유형', '특정 시술 유형', '시술 당시 나이', '난자 출처', '정자 출처', '배아 생성 주요 이유']:
    print('\n' + '=' * 80)
    print(col)
    display_group_rate(train, col, cfg.target)


#### 01-5. 결과 해석

시술 유형, 나이, 난자/정자 출처, 배아 생성 이유는 성공률 차이를 보일 가능성이 큽니다.  
이 변수들은 원본 범주형으로도 쓰고, 일부는 조합 변수로도 확장합니다.


### 01-6. 나이·난자·배아 관련 구간화 기준 탐색


In [ ]:
# 나이 구간과 수량형 구간 확인

AGE_ORD = {
    '만18-34세': 0,
    '만35-37세': 1,
    '만38-39세': 2,
    '만40-42세': 3,
    '만43-44세': 4,
    '만45-50세': 5,
    '알 수 없음': np.nan,
}

AGE_MID = {
    '만18-34세': 29.0,
    '만35-37세': 36.0,
    '만38-39세': 38.5,
    '만40-42세': 41.0,
    '만43-44세': 43.5,
    '만45-50세': 47.5,
    '알 수 없음': np.nan,
}

if '시술 당시 나이' in train.columns:
    age_tmp = train[[cfg.target, '시술 당시 나이']].copy()
    age_tmp['age_ord'] = age_tmp['시술 당시 나이'].map(AGE_ORD)
    display(age_tmp.groupby('시술 당시 나이', dropna=False)[cfg.target].agg(count='count', success_rate='mean'))


def count_bin(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors='coerce')
    return pd.cut(values, bins=[-np.inf, 0, 1, 2, 5, 10, np.inf], labels=['0', '1', '2', '3-5', '6-10', '11+'])

for col in ['총 생성 배아 수', '이식된 배아 수', '저장된 배아 수', '미세주입된 난자 수', '수집된 신선 난자 수']:
    if col in train.columns:
        tmp = train[[cfg.target, col]].copy()
        tmp[f'{col}_bin'] = count_bin(tmp[col])
        print('\n' + col)
        display(tmp.groupby(f'{col}_bin', dropna=False)[cfg.target].agg(count='count', success_rate='mean'))


#### 01-6. 결과 해석

나이와 수량형 변수는 원본 값을 그대로 쓰는 것만으로는 패턴이 잘 드러나지 않을 수 있습니다.  
그래서 나이는 순서형과 38세/40세/43세 이상 flag로, 배아·난자 수는 구간형 변수로 추가합니다.


### 01-7. 이상치 후보 확인


In [ ]:
# 수치형 변수의 기초 통계 및 IQR 이상치 후보 확인

numeric_raw_cols = train.drop(columns=[cfg.target], errors='ignore').select_dtypes(include=[np.number]).columns.tolist()
numeric_raw_cols = [col for col in numeric_raw_cols if col != cfg.id_col]

if numeric_raw_cols:
    desc = train[numeric_raw_cols].describe().T
    display(desc.head(30))

    outlier_rows = []
    for col in numeric_raw_cols:
        s = pd.to_numeric(train[col], errors='coerce').dropna()
        if s.nunique() <= 2 or len(s) < 30:
            continue
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outlier_rate = ((s < lower) | (s > upper)).mean()
        outlier_rows.append({'column': col, 'lower': lower, 'upper': upper, 'outlier_rate': outlier_rate})
    outlier_summary = pd.DataFrame(outlier_rows).sort_values('outlier_rate', ascending=False)
    display(outlier_summary.head(30))
else:
    outlier_summary = pd.DataFrame()
    print('수치형 원본 컬럼이 없습니다.')


#### 01-7. 결과 해석

트리 기반 모델은 이상치에 비교적 강하지만, 극단적인 값은 split 기준을 흔들 수 있습니다.  
따라서 수치형 feature에는 train 기준 quantile clipping을 fold 내부에서 적용합니다.


### 01-8. 수치형 변수 간 상관관계 확인


In [ ]:
# target과 상관관계가 높은 수치형 변수 확인

if numeric_raw_cols:
    corr_df = train[numeric_raw_cols + [cfg.target]].corr(numeric_only=True)[cfg.target].drop(cfg.target).abs().sort_values(ascending=False)
    display(corr_df.head(30).to_frame('abs_corr_with_target'))
else:
    print('상관관계를 계산할 수치형 컬럼이 없습니다.')


#### 01-8. 결과 해석

상관관계는 선형 관계만 보여주므로 절대적인 feature 선택 기준은 아닙니다.  
다만 배아/난자 관련 수치형 변수의 방향성을 빠르게 확인하고, 비율 변수 후보를 정할 때 참고합니다.


### 01-9. EDA 종합 정리 및 전처리 방향 제시

EDA에서 얻은 전처리 방향은 다음과 같습니다.

1. target 불균형이 있으므로 OOF ROC-AUC 중심으로 검증합니다.  
2. 시술 유형과 나이는 핵심 범주형 변수로 유지합니다.  
3. 배아·난자·이식 관련 결측은 구조적 결측 flag로 반영합니다.  
4. 수량형 변수는 원본, 결측 flag, zero flag, 구간형 변수를 함께 둡니다.  
5. 시술 유형 × 나이, 나이 × 난자 출처 같은 조합 변수는 제한적으로 추가합니다.


## 02. 데이터 전처리 및 Feature Engineering


### 02-1. 학습용 데이터 구성


In [ ]:
# 학습용 feature와 target 분리

y = train[cfg.target].astype(int).values
base_feature_cols = [col for col in train.columns if col not in [cfg.target] and col in test.columns]
print('base feature count:', len(base_feature_cols))
print(base_feature_cols[:20])


#### 02-1. 처리 방식 설명

train과 test에 공통으로 존재하는 컬럼만 feature 후보로 사용합니다.  
ID는 나중에 제외하지만, 제출 파일 검증을 위해 원본에는 유지합니다.


### 02-2. 결측치 처리: 구조적 결측 및 일반 결측 flag 생성


In [ ]:
# Feature Engineering 상수와 유틸리티

COUNT_COLS = [
    '총 시술 횟수', '클리닉 내 총 시술 횟수', 'IVF 시술 횟수', 'DI 시술 횟수',
    '총 임신 횟수', 'IVF 임신 횟수', 'DI 임신 횟수',
    '총 출산 횟수', 'IVF 출산 횟수', 'DI 출산 횟수',
]

GENETIC_TEST_COLS = ['PGD 시술 여부', 'PGS 시술 여부', '착상 전 유전 검사 사용 여부']


def safe_str(df: pd.DataFrame, col: str, token: str = '__MISSING__') -> pd.Series:
    if col not in df.columns:
        return pd.Series(token, index=df.index, dtype='string')
    return df[col].astype('string').fillna(token)


def safe_num(df: pd.DataFrame, col: str) -> pd.Series:
    if col not in df.columns:
        return pd.Series(np.nan, index=df.index, dtype='float64')
    return pd.to_numeric(df[col], errors='coerce')


def contains_token(series: pd.Series, token: str) -> pd.Series:
    return series.astype('string').str.contains(token, case=False, regex=False, na=False)


def ratio(num: pd.Series, den: pd.Series) -> pd.Series:
    num = pd.to_numeric(num, errors='coerce')
    den = pd.to_numeric(den, errors='coerce').replace(0, np.nan)
    out = num / den
    return out.replace([np.inf, -np.inf], np.nan)


def count_to_ord(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors='coerce')
    extracted = s.astype('string').str.extract(r'(\d+)')[0]
    return pd.to_numeric(extracted, errors='coerce')


def bin_count(s: pd.Series) -> pd.Series:
    values = pd.to_numeric(s, errors='coerce')
    return pd.cut(
        values,
        bins=[-np.inf, 0, 1, 2, 5, 10, np.inf],
        labels=['0', '1', '2', '3-5', '6-10', '11+'],
    ).astype('string')


In [ ]:
# 구조적 결측 flag와 일반 결측 flag를 포함한 feature 생성 함수


def add_features(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    df = df.copy()

    treatment = safe_str(df, '시술 유형', cfg.missing_token)
    procedure = safe_str(df, '특정 시술 유형', cfg.missing_token)
    age = safe_str(df, '시술 당시 나이', cfg.missing_token)
    egg_source = safe_str(df, '난자 출처', cfg.missing_token)
    sperm_source = safe_str(df, '정자 출처', cfg.missing_token)
    reason = safe_str(df, '배아 생성 주요 이유', cfg.missing_token)

    # 시술 구조 flag
    df['f_is_ivf'] = (treatment == 'IVF').astype('int8')
    df['f_is_di'] = (treatment == 'DI').astype('int8')
    df['f_has_icsi'] = contains_token(procedure, 'ICSI').astype('int8')
    df['f_has_ivf_token'] = contains_token(procedure, 'IVF').astype('int8')
    df['f_has_iui'] = contains_token(procedure, 'IUI').astype('int8')
    df['f_has_ah'] = contains_token(procedure, 'AH').astype('int8')
    df['f_has_blastocyst'] = contains_token(procedure, 'BLASTOCYST').astype('int8')
    df['f_proc_missing'] = (procedure == cfg.missing_token).astype('int8')

    # 나이 feature
    df['f_age_ord'] = age.map(AGE_ORD).astype('float32')
    df['f_age_mid'] = age.map(AGE_MID).astype('float32')
    df['f_age_missing'] = df['f_age_ord'].isna().astype('int8')
    df['f_age_38plus'] = (df['f_age_ord'] >= 2).fillna(False).astype('int8')
    df['f_age_40plus'] = (df['f_age_ord'] >= 3).fillna(False).astype('int8')
    df['f_age_43plus'] = (df['f_age_ord'] >= 4).fillna(False).astype('int8')

    # 문자열 횟수형 정리
    for col in COUNT_COLS:
        if col in df.columns:
            df[f'f_{col}_ord'] = count_to_ord(df[col]).astype('float32')
            df[f'f_{col}_missing'] = df[col].isna().astype('int8')

    # 배아/난자 관련 결측 구조
    present_cols = [col for col in EMBRYO_COLS if col in df.columns]
    if present_cols:
        embryo_missing = df[present_cols].isna()
        df['f_embryo_missing_count'] = embryo_missing.sum(axis=1).astype('int16')
        df['f_embryo_all_missing'] = (df['f_embryo_missing_count'] == len(present_cols)).astype('int8')
        df['f_embryo_any_present'] = (df['f_embryo_missing_count'] < len(present_cols)).astype('int8')
        df['f_embryo_missing_rate'] = (df['f_embryo_missing_count'] / len(present_cols)).astype('float32')

    for col in present_cols:
        s = safe_num(df, col)
        df[f'f_{col}_missing'] = s.isna().astype('int8')
        df[f'f_{col}_is_zero'] = (s.fillna(0) == 0).astype('int8')
        df[f'f_{col}_bin'] = bin_count(s).fillna(cfg.missing_token)

    # 고결측 유전검사 계열은 known/yes 여부로 보조 반영
    for col in GENETIC_TEST_COLS:
        if col in df.columns:
            s = safe_str(df, col, cfg.missing_token)
            df[f'f_{col}_known'] = (s != cfg.missing_token).astype('int8')
            df[f'f_{col}_yes'] = s.str.contains('1|예|Yes|TRUE|True', regex=True, na=False).astype('int8')

    # 비율 변수
    created = safe_num(df, '총 생성 배아 수')
    transferred = safe_num(df, '이식된 배아 수')
    saved = safe_num(df, '저장된 배아 수')
    thawed = safe_num(df, '해동된 배아 수')
    icsi_egg = safe_num(df, '미세주입된 난자 수')
    icsi_embryo = safe_num(df, '미세주입에서 생성된 배아 수')
    fresh_egg = safe_num(df, '수집된 신선 난자 수')
    mixed_egg = safe_num(df, '혼합된 난자 수')

    df['f_transfer_per_created'] = ratio(transferred, created).astype('float32')
    df['f_saved_per_created'] = ratio(saved, created).astype('float32')
    df['f_thawed_per_transferred'] = ratio(thawed, transferred).astype('float32')
    df['f_icsi_embryo_per_icsi_egg'] = ratio(icsi_embryo, icsi_egg).astype('float32')
    df['f_mixed_per_fresh_egg'] = ratio(mixed_egg, fresh_egg).astype('float32')

    # 조합 범주형 변수
    df['f_treatment_age'] = treatment + '__' + age
    df['f_treatment_procedure'] = treatment + '__' + procedure
    df['f_age_egg_source'] = age + '__' + egg_source
    df['f_egg_sperm_source'] = egg_source + '__' + sperm_source
    df['f_reason_procedure'] = reason + '__' + procedure

    return df


#### 02-2. 결과 해석

결측은 제거 대상이 아니라 모델에 전달할 정보입니다. 특히 배아·난자 계열의 동시 결측은 `f_embryo_missing_count`와 `f_embryo_all_missing`으로 요약합니다.


### 02-3. 이상치 처리


In [ ]:
# fold 내부에서 사용할 clipping 유틸리티


def fit_clip_bounds(X_train: pd.DataFrame, num_cols: List[str], lower_q: float = 0.005, upper_q: float = 0.995) -> Dict[str, Tuple[float, float]]:
    bounds = {}
    for col in num_cols:
        s = pd.to_numeric(X_train[col], errors='coerce')
        if s.notna().sum() < 30:
            continue
        lo = s.quantile(lower_q)
        hi = s.quantile(upper_q)
        if pd.notna(lo) and pd.notna(hi) and lo < hi:
            bounds[col] = (float(lo), float(hi))
    return bounds


def apply_clip_bounds(X_df: pd.DataFrame, bounds: Dict[str, Tuple[float, float]]) -> pd.DataFrame:
    X_df = X_df.copy()
    for col, (lo, hi) in bounds.items():
        if col in X_df.columns:
            X_df[col] = pd.to_numeric(X_df[col], errors='coerce').clip(lo, hi)
    return X_df


#### 02-3. 결과 해석

clipping 기준은 전체 train이 아니라 fold 내부 train에서만 계산합니다. validation/test 정보가 clipping 기준에 섞이지 않도록 하기 위함입니다.


### 02-4. 임상 지식 기반 파생변수 생성


In [ ]:
# feature 생성 실행

train_fe = add_features(train, cfg)
test_fe = add_features(test, cfg)

print('train_fe:', train_fe.shape)
print('test_fe:', test_fe.shape)
new_features = sorted(set(train_fe.columns) - set(train.columns))
print('added feature count:', len(new_features))
print(new_features[:50])


#### 02-4. 결과 해석

시술 구조, 나이, 배아·난자 결측, 비율, 조합 변수를 추가했습니다.  
이 변수들은 모두 test의 target 없이 생성 가능하므로 제출 단계에서도 동일하게 적용할 수 있습니다.


### 02-5. 나이·난자·배아 구간화 변수 생성


In [ ]:
# 생성된 구간화 변수 확인

bin_cols = [col for col in train_fe.columns if col.endswith('_bin')]
print('bin feature count:', len(bin_cols))
print(bin_cols[:30])

for col in bin_cols[:5]:
    print('\n' + col)
    display(train_fe.groupby(col, dropna=False)[cfg.target].agg(count='count', success_rate='mean'))


#### 02-5. 결과 해석

구간화 변수는 트리 모델이 수량형 변수의 비선형 구간 패턴을 더 쉽게 잡도록 돕는 보조 feature입니다.


### 02-6. 조합 변수 및 비율 변수 생성


In [ ]:
# 조합 변수와 비율 변수 확인

combo_cols = [col for col in train_fe.columns if col.startswith('f_') and train_fe[col].dtype.name in ['object', 'string', 'category']]
ratio_cols = [col for col in train_fe.columns if 'per_' in col]

print('combo/categorical added sample:', combo_cols[:20])
print('ratio cols:', ratio_cols)

display(train_fe[ratio_cols].describe().T if ratio_cols else pd.DataFrame())


#### 02-6. 결과 해석

조합 변수는 너무 많이 만들면 과적합과 cardinality 문제가 생길 수 있습니다.  
따라서 v5 흐름과 비슷하게 핵심 조합만 남겼습니다.


### 02-7. 범주형/수치형 변수 정리


In [ ]:
# 모델 입력 feature 정리

feature_cols = [
    col for col in train_fe.columns
    if col not in [cfg.target, cfg.id_col]
    and col in test_fe.columns
]

X = train_fe[feature_cols].copy()
X_test = test_fe[feature_cols].copy()
y = train_fe[cfg.target].astype(int).values

cat_cols = [col for col in feature_cols if str(X[col].dtype) in ['object', 'string', 'category']]
num_cols = [col for col in feature_cols if col not in cat_cols]

print('feature count:', len(feature_cols))
print('categorical count:', len(cat_cols))
print('numeric count:', len(num_cols))
print('categorical sample:', cat_cols[:20])
print('numeric sample:', num_cols[:20])

feature_manifest = {
    'feature_count': len(feature_cols),
    'categorical_count': len(cat_cols),
    'numeric_count': len(num_cols),
    'categorical_columns': cat_cols,
    'numeric_columns': num_cols,
}
with open(cfg.out_dir / 'feature_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(feature_manifest, f, ensure_ascii=False, indent=2)


#### 02-7. 결과 해석

CatBoost는 범주형을 문자열로 유지해 사용하고, LightGBM/XGBoost/sklearn baseline은 fold 내부 train 기준 label/frequency encoding을 적용합니다.


### 02-8. 인코딩 전략


In [ ]:
# CatBoost용 / 일반 모델용 전처리 함수


def clip_pred(pred: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    return np.clip(np.asarray(pred, dtype='float64'), eps, 1 - eps)


def make_catboost_frames(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    X_test: pd.DataFrame,
    cat_cols: List[str],
    num_cols: List[str],
    cfg: Config,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, List[int]]:
    cols = num_cols + cat_cols
    X_tr = X_train[cols].copy()
    X_va = X_valid[cols].copy()
    X_te = X_test[cols].copy()

    bounds = fit_clip_bounds(X_tr, num_cols)
    X_tr = apply_clip_bounds(X_tr, bounds)
    X_va = apply_clip_bounds(X_va, bounds)
    X_te = apply_clip_bounds(X_te, bounds)

    for col in num_cols:
        median = pd.to_numeric(X_tr[col], errors='coerce').median()
        if pd.isna(median):
            median = 0.0
        X_tr[col] = pd.to_numeric(X_tr[col], errors='coerce').fillna(median).astype('float32')
        X_va[col] = pd.to_numeric(X_va[col], errors='coerce').fillna(median).astype('float32')
        X_te[col] = pd.to_numeric(X_te[col], errors='coerce').fillna(median).astype('float32')

    for col in cat_cols:
        X_tr[col] = X_tr[col].astype('string').fillna(cfg.missing_token).astype(str)
        X_va[col] = X_va[col].astype('string').fillna(cfg.missing_token).astype(str)
        X_te[col] = X_te[col].astype('string').fillna(cfg.missing_token).astype(str)

    cat_indices = [cols.index(col) for col in cat_cols]
    return X_tr, X_va, X_te, cat_indices


def make_encoded_frames(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    X_test: pd.DataFrame,
    cat_cols: List[str],
    num_cols: List[str],
    cfg: Config,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    X_tr = pd.DataFrame(index=X_train.index)
    X_va = pd.DataFrame(index=X_valid.index)
    X_te = pd.DataFrame(index=X_test.index)

    bounds = fit_clip_bounds(X_train, num_cols)
    X_train_c = apply_clip_bounds(X_train, bounds)
    X_valid_c = apply_clip_bounds(X_valid, bounds)
    X_test_c = apply_clip_bounds(X_test, bounds)

    for col in num_cols:
        median = pd.to_numeric(X_train_c[col], errors='coerce').median()
        if pd.isna(median):
            median = 0.0
        X_tr[col] = pd.to_numeric(X_train_c[col], errors='coerce').fillna(median).astype('float32')
        X_va[col] = pd.to_numeric(X_valid_c[col], errors='coerce').fillna(median).astype('float32')
        X_te[col] = pd.to_numeric(X_test_c[col], errors='coerce').fillna(median).astype('float32')

    for col in cat_cols:
        s_tr = X_train[col].astype('string').fillna(cfg.missing_token)
        s_va = X_valid[col].astype('string').fillna(cfg.missing_token)
        s_te = X_test[col].astype('string').fillna(cfg.missing_token)

        counts = s_tr.value_counts(dropna=False)
        label_map = {value: idx + 1 for idx, value in enumerate(counts.index.tolist())}
        freq_map = (counts / max(len(s_tr), 1)).to_dict()

        X_tr[f'{col}__label'] = s_tr.map(label_map).fillna(0).astype('float32')
        X_va[f'{col}__label'] = s_va.map(label_map).fillna(0).astype('float32')
        X_te[f'{col}__label'] = s_te.map(label_map).fillna(0).astype('float32')

        X_tr[f'{col}__freq'] = s_tr.map(freq_map).fillna(0).astype('float32')
        X_va[f'{col}__freq'] = s_va.map(freq_map).fillna(0).astype('float32')
        X_te[f'{col}__freq'] = s_te.map(freq_map).fillna(0).astype('float32')

    return X_tr, X_va, X_te


#### 02-8. 결과 해석

인코딩과 결측 대체는 반드시 fold 내부 train 기준으로만 fit합니다.  
이 기준을 지키면 validation fold와 test 정보가 학습 과정에 섞이는 문제를 줄일 수 있습니다.


### 02-9. 스케일링 전략


In [ ]:
# 선형 baseline용 스케일링은 fold 내부에서 pipeline으로 처리합니다.
# 트리 기반 모델은 별도 scaling 없이 사용할 수 있습니다.

print('LogisticRegression baseline에만 StandardScaler를 적용합니다.')
print('Boosting/Tree 계열 모델은 scaling 없이 encoded frame을 사용합니다.')


#### 02-9. 결과 해석

스케일링은 선형 모델에는 필요하지만, CatBoost/LightGBM/XGBoost 같은 트리 기반 모델에는 필수는 아닙니다.


### 02-10. 학습/검증 데이터 분리 및 K-Fold 설정


In [ ]:
# K-Fold 설정

skf_preview = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.random_state)
fold_sizes = []
for fold, (_, va_idx) in enumerate(skf_preview.split(X, y), start=1):
    fold_sizes.append({'fold': fold, 'valid_size': len(va_idx), 'valid_target_mean': float(y[va_idx].mean())})
fold_preview = pd.DataFrame(fold_sizes)
display(fold_preview)


#### 02-10. 결과 해석

StratifiedKFold를 사용해 각 fold의 target 비율이 크게 흔들리지 않도록 합니다.  
이후 모든 주요 모델은 같은 방식의 OOF 검증으로 비교합니다.


## 03. 모델링


### 03-1. Baseline 모델 성능 비교


In [ ]:
# Baseline 모델 평가 함수


def evaluate_baselines(X: pd.DataFrame, y: np.ndarray, X_test: pd.DataFrame, cat_cols: List[str], num_cols: List[str], cfg: Config) -> pd.DataFrame:
    baseline_models = {
        'logistic': lambda seed: make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=500, random_state=seed, n_jobs=1)
        ),
        'random_forest': lambda seed: RandomForestClassifier(
            n_estimators=60 if cfg.run_mode == 'quick' else 400,
            max_depth=8,
            min_samples_leaf=20,
            random_state=seed,
            n_jobs=1,
        ),
        'gradient_boosting': lambda seed: GradientBoostingClassifier(
            n_estimators=80 if cfg.run_mode == 'quick' else 400,
            learning_rate=0.05,
            max_depth=3,
            random_state=seed,
        ),
    }

    rows = []
    skf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.random_state)
    for model_name, factory in baseline_models.items():
        oof = np.zeros(len(X), dtype='float64')
        for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
            X_tr, X_va, _ = make_encoded_frames(X.iloc[tr_idx], X.iloc[va_idx], X_test, cat_cols, num_cols, cfg)
            y_tr, y_va = y[tr_idx], y[va_idx]
            model = factory(cfg.random_state + fold)
            model.fit(X_tr, y_tr)
            pred = clip_pred(model.predict_proba(X_va)[:, 1])
            oof[va_idx] = pred
        rows.append({'model_name': model_name, 'oof_auc': roc_auc_score(y, oof), 'oof_logloss': log_loss(y, oof)})
        print(model_name, rows[-1])
    return pd.DataFrame(rows).sort_values('oof_auc', ascending=False).reset_index(drop=True)

baseline_score_df = evaluate_baselines(X, y, X_test, cat_cols, num_cols, cfg)
display(baseline_score_df)
baseline_score_df.to_csv(cfg.out_dir / 'baseline_scores.csv', index=False)


#### 03-1. 결과 해석

Baseline은 복잡한 모델의 성능이 충분히 의미 있는지 확인하기 위한 기준선입니다.  
최종 제출 후보는 boosting 모델과 앙상블을 중심으로 만들지만, baseline 점수는 보고서의 기준점으로 남겨둡니다.


### 모델 학습 공통 함수


In [ ]:
# Boosting 모델 공통 함수


def available_model_names() -> List[str]:
    names = []
    if CatBoostClassifier is not None:
        names.append('catboost')
    if lgb is not None:
        names.append('lightgbm')
    if XGBClassifier is not None:
        names.append('xgboost')
    if not names:
        names.append('sklearn_gb')
    return names


def make_model(model_name: str, seed: int, cfg: Config):
    if model_name == 'catboost':
        return CatBoostClassifier(
            loss_function='Logloss',
            eval_metric='AUC',
            iterations=cfg.catboost_iterations(),
            learning_rate=0.035 if cfg.run_mode == 'full' else 0.06,
            depth=7,
            l2_leaf_reg=6,
            random_seed=seed,
            task_type='GPU' if cfg.use_catboost_gpu else 'CPU',
            verbose=False,
            allow_writing_files=False,
        )
    if model_name == 'lightgbm':
        return lgb.LGBMClassifier(
            objective='binary',
            n_estimators=cfg.lightgbm_estimators(),
            learning_rate=0.025 if cfg.run_mode == 'full' else 0.05,
            num_leaves=63,
            max_depth=-1,
            min_child_samples=35,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.2,
            reg_lambda=1.0,
            random_state=seed,
            n_jobs=-1,
            verbosity=-1,
        )
    if model_name == 'xgboost':
        return XGBClassifier(
            n_estimators=cfg.xgboost_estimators(),
            learning_rate=0.03 if cfg.run_mode == 'full' else 0.05,
            max_depth=5,
            min_child_weight=8,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.2,
            reg_lambda=1.0,
            objective='binary:logistic',
            eval_metric='auc',
            tree_method='hist',
            random_state=seed,
            n_jobs=-1,
        )
    if model_name == 'sklearn_gb':
        return GradientBoostingClassifier(
            n_estimators=100 if cfg.run_mode == 'quick' else 500,
            learning_rate=0.04,
            max_depth=3,
            random_state=seed,
        )
    raise ValueError(f'unknown model: {model_name}')


def train_oof_model(
    model_name: str,
    seed: int,
    X: pd.DataFrame,
    y: np.ndarray,
    X_test: pd.DataFrame,
    cat_cols: List[str],
    num_cols: List[str],
    cfg: Config,
) -> Dict[str, object]:
    skf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(X), dtype='float64')
    test_pred = np.zeros(len(X_test), dtype='float64')
    fold_rows = []
    feature_importances = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        X_tr_raw = X.iloc[tr_idx]
        X_va_raw = X.iloc[va_idx]
        y_tr = y[tr_idx]
        y_va = y[va_idx]
        model = make_model(model_name, seed + fold, cfg)

        if model_name == 'catboost':
            X_tr, X_va, X_te, cat_indices = make_catboost_frames(X_tr_raw, X_va_raw, X_test, cat_cols, num_cols, cfg)
            train_pool = Pool(X_tr, y_tr, cat_features=cat_indices)
            valid_pool = Pool(X_va, y_va, cat_features=cat_indices)
            test_pool = Pool(X_te, cat_features=cat_indices)
            model.fit(train_pool, eval_set=valid_pool, use_best_model=True, verbose=False)
            va_pred = model.predict_proba(valid_pool)[:, 1]
            te_pred = model.predict_proba(test_pool)[:, 1]
            try:
                fi = pd.DataFrame({'feature': X_tr.columns, 'importance': model.get_feature_importance()})
                feature_importances.append(fi)
            except Exception:
                pass
        else:
            X_tr, X_va, X_te = make_encoded_frames(X_tr_raw, X_va_raw, X_test, cat_cols, num_cols, cfg)
            if model_name == 'lightgbm':
                model.fit(
                    X_tr,
                    y_tr,
                    eval_set=[(X_va, y_va)],
                    eval_metric='auc',
                    callbacks=[lgb.early_stopping(120, verbose=False), lgb.log_evaluation(0)],
                )
            elif model_name == 'xgboost':
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            else:
                model.fit(X_tr, y_tr)
            va_pred = model.predict_proba(X_va)[:, 1]
            te_pred = model.predict_proba(X_te)[:, 1]
            if hasattr(model, 'feature_importances_'):
                fi = pd.DataFrame({'feature': X_tr.columns, 'importance': model.feature_importances_})
                feature_importances.append(fi)

        va_pred = clip_pred(va_pred)
        te_pred = clip_pred(te_pred)
        oof[va_idx] = va_pred
        test_pred += te_pred / cfg.n_splits
        fold_auc = roc_auc_score(y_va, va_pred)
        fold_logloss = log_loss(y_va, va_pred)
        fold_rows.append({'fold': fold, 'auc': fold_auc, 'logloss': fold_logloss})
        print(f'{model_name} seed={seed} fold={fold} auc={fold_auc:.6f} logloss={fold_logloss:.6f}')

    oof = clip_pred(oof)
    test_pred = clip_pred(test_pred)
    fi_summary = pd.DataFrame()
    if feature_importances:
        fi_summary = (
            pd.concat(feature_importances)
            .groupby('feature', as_index=False)['importance'].mean()
            .sort_values('importance', ascending=False)
        )
    result = {
        'run_name': f'{model_name}_s{seed}',
        'model_name': model_name,
        'seed': seed,
        'oof': oof,
        'test_pred': test_pred,
        'auc': roc_auc_score(y, oof),
        'logloss': log_loss(y, oof),
        'folds': pd.DataFrame(fold_rows),
        'feature_importance': fi_summary,
    }
    print(f"[OOF] {result['run_name']} auc={result['auc']:.6f} logloss={result['logloss']:.6f}")
    return result

prediction_bank: Dict[str, Dict[str, object]] = {}
score_rows: List[Dict[str, object]] = []
print('available models:', available_model_names())


### 03-2. CatBoost 모델 학습


In [ ]:
# CatBoost 5-Fold OOF 검증

if 'catboost' in available_model_names():
    for seed in cfg.seeds():
        result = train_oof_model('catboost', seed, X, y, X_test, cat_cols, num_cols, cfg)
        prediction_bank[result['run_name']] = result
        score_rows.append({
            'run_name': result['run_name'],
            'model_name': result['model_name'],
            'seed': result['seed'],
            'oof_auc': result['auc'],
            'oof_logloss': result['logloss'],
        })
else:
    print('CatBoost가 설치되어 있지 않아 건너뜁니다.')


#### 03-2. 결과 해석

CatBoost는 범주형 변수를 직접 처리할 수 있어 이 데이터의 강한 기준 모델로 적합합니다.


### 03-3. LightGBM 모델 학습


In [ ]:
# LightGBM 5-Fold OOF 검증

if 'lightgbm' in available_model_names():
    for seed in cfg.seeds():
        result = train_oof_model('lightgbm', seed, X, y, X_test, cat_cols, num_cols, cfg)
        prediction_bank[result['run_name']] = result
        score_rows.append({
            'run_name': result['run_name'],
            'model_name': result['model_name'],
            'seed': result['seed'],
            'oof_auc': result['auc'],
            'oof_logloss': result['logloss'],
        })
else:
    print('LightGBM이 설치되어 있지 않아 건너뜁니다.')


#### 03-3. 결과 해석

LightGBM은 빠르고 안정적인 boosting 모델입니다. CatBoost와 예측 성향이 조금 달라 앙상블 재료로 유용합니다.


### 03-4. XGBoost 모델 학습


In [ ]:
# XGBoost 5-Fold OOF 검증

if 'xgboost' in available_model_names():
    for seed in cfg.seeds():
        result = train_oof_model('xgboost', seed, X, y, X_test, cat_cols, num_cols, cfg)
        prediction_bank[result['run_name']] = result
        score_rows.append({
            'run_name': result['run_name'],
            'model_name': result['model_name'],
            'seed': result['seed'],
            'oof_auc': result['auc'],
            'oof_logloss': result['logloss'],
        })
else:
    print('XGBoost가 설치되어 있지 않아 건너뜁니다.')

# boosting 라이브러리가 하나도 없는 환경용 fallback
if not prediction_bank and 'sklearn_gb' in available_model_names():
    for seed in cfg.seeds():
        result = train_oof_model('sklearn_gb', seed, X, y, X_test, cat_cols, num_cols, cfg)
        prediction_bank[result['run_name']] = result
        score_rows.append({
            'run_name': result['run_name'],
            'model_name': result['model_name'],
            'seed': result['seed'],
            'oof_auc': result['auc'],
            'oof_logloss': result['logloss'],
        })

score_df = pd.DataFrame(score_rows).sort_values(['oof_auc', 'oof_logloss'], ascending=[False, True]).reset_index(drop=True)
display(score_df)
score_df.to_csv(cfg.out_dir / 'model_scores.csv', index=False)


#### 03-4. 결과 해석

XGBoost는 CatBoost/LightGBM과 다른 예측 분포를 줄 수 있어 3-model 앙상블 후보로 확인합니다.  
단, OOF가 낮으면 억지로 포함하지 않습니다.


### 03-5. 모델 성능 비교 시각화


In [ ]:
# 모델 성능 비교 시각화

if score_df.empty:
    raise RuntimeError('학습된 모델이 없습니다. 이전 셀 로그를 확인하세요.')

if plt is not None:
    plot_df = score_df.sort_values('oof_auc', ascending=True)
    ax = plot_df.plot.barh(x='run_name', y='oof_auc', legend=False, title='OOF ROC-AUC by model')
    ax.set_xlabel('OOF ROC-AUC')
    plt.show()

    plot_df = score_df.sort_values('oof_logloss', ascending=False)
    ax = plot_df.plot.barh(x='run_name', y='oof_logloss', legend=False, title='OOF LogLoss by model')
    ax.set_xlabel('OOF LogLoss')
    plt.show()
else:
    display(score_df)


#### 03-5. 결과 해석

OOF ROC-AUC가 가장 높은 모델을 우선 확인합니다. 다만 상위 모델 간 차이가 작다면 앙상블 결과를 함께 봐야 합니다.


### 03-6. Feature Importance 확인


In [ ]:
# Feature Importance 확인

fi_tables = []
for name, result in prediction_bank.items():
    fi = result.get('feature_importance')
    if isinstance(fi, pd.DataFrame) and not fi.empty:
        tmp = fi.copy()
        tmp['run_name'] = name
        fi_tables.append(tmp)

if fi_tables:
    feature_importance_df = (
        pd.concat(fi_tables)
        .groupby('feature', as_index=False)['importance'].mean()
        .sort_values('importance', ascending=False)
    )
    display(feature_importance_df.head(40))
    feature_importance_df.to_csv(cfg.out_dir / 'feature_importance.csv', index=False)
else:
    feature_importance_df = pd.DataFrame()
    print('사용 가능한 feature importance가 없습니다.')


#### 03-6. Feature Importance 결과 해석

Feature importance는 인과관계를 의미하지 않습니다.  
하지만 나이, 시술 유형, 구조적 결측, 배아/난자 관련 변수가 상위에 나타나는지 확인하면 feature 설계가 EDA 방향과 맞는지 점검할 수 있습니다.


### 03-7. Ensemble 후보 비교


In [ ]:
# Ensemble 유틸리티


def get_oof(name: str) -> np.ndarray:
    return np.asarray(prediction_bank[name]['oof'], dtype='float64')


def get_test_pred(name: str) -> np.ndarray:
    return np.asarray(prediction_bank[name]['test_pred'], dtype='float64')


def weighted_average(names: List[str], weights: np.ndarray, use_rank: bool = False) -> Tuple[np.ndarray, np.ndarray]:
    weights = np.asarray(weights, dtype='float64')
    weights = weights / weights.sum()
    if use_rank:
        oof_mat = np.column_stack([rankdata(get_oof(name)) / len(y) for name in names])
        test_mat = np.column_stack([rankdata(get_test_pred(name)) / len(X_test) for name in names])
    else:
        oof_mat = np.column_stack([get_oof(name) for name in names])
        test_mat = np.column_stack([get_test_pred(name) for name in names])
    return clip_pred(oof_mat @ weights), clip_pred(test_mat @ weights)


def add_blend_row(rows: List[Dict[str, object]], names: List[str], weights: Iterable[float], kind: str) -> None:
    weights = np.asarray(list(weights), dtype='float64')
    weights = weights / weights.sum()
    oof_blend, _ = weighted_average(names, weights, use_rank=(kind == 'rank'))
    rows.append({
        'blend_name': f'{kind}__' + '__'.join(names),
        'kind': kind,
        'members': '|'.join(names),
        'weights': '|'.join(f'{w:.8f}' for w in weights),
        'oof_auc': roc_auc_score(y, oof_blend),
        'oof_logloss': log_loss(y, oof_blend),
    })


In [ ]:
# 단일 모델 및 2-model raw ensemble 후보 비교

names_sorted = score_df['run_name'].tolist()
blend_rows: List[Dict[str, object]] = []

for name in names_sorted:
    add_blend_row(blend_rows, [name], [1.0], kind='raw')

for a, b in combinations(names_sorted, 2):
    for w in np.linspace(0.05, 0.95, 19):
        add_blend_row(blend_rows, [a, b], [w, 1 - w], kind='raw')

raw_blend_df = (
    pd.DataFrame(blend_rows)
    .drop_duplicates(['kind', 'members', 'weights'])
    .sort_values(['oof_auc', 'oof_logloss'], ascending=[False, True])
    .reset_index(drop=True)
)
display(raw_blend_df.head(30))


#### 03-7. 결과 해석

2-model raw ensemble은 가장 기본적인 앙상블입니다.  
CatBoost와 LightGBM처럼 성향이 다른 모델을 적절히 섞었을 때 단일 모델보다 좋아질 수 있습니다.


### 03-8. 세밀 가중치 탐색


In [ ]:
# 상위 2개 모델에 대한 세밀 가중치 탐색

fine_rows = []
if len(names_sorted) >= 2:
    a, b = names_sorted[:2]
    for w in np.linspace(0.01, 0.99, 99):
        add_blend_row(fine_rows, [a, b], [w, 1 - w], kind='raw')
    fine_blend_df = (
        pd.DataFrame(fine_rows)
        .sort_values(['oof_auc', 'oof_logloss'], ascending=[False, True])
        .reset_index(drop=True)
    )
    display(fine_blend_df.head(20))
else:
    fine_blend_df = pd.DataFrame()
    print('세밀 가중치 탐색을 하려면 모델이 2개 이상 필요합니다.')


#### 03-8. 결과 해석

세밀 탐색은 상위 2개 모델에만 적용합니다.  
전체 모델 조합을 과하게 뒤지는 대신, 보고서 흐름상 납득 가능한 범위에서만 확인합니다.


### 03-9. Rank Average 앙상블


In [ ]:
# Rank Average 앙상블

rank_rows = []
for a, b in combinations(names_sorted, 2):
    for w in np.linspace(0.05, 0.95, 19):
        add_blend_row(rank_rows, [a, b], [w, 1 - w], kind='rank')

if len(names_sorted) >= 3:
    top3 = names_sorted[:3]
    add_blend_row(rank_rows, top3, [1, 1, 1], kind='rank')
    aucs = score_df.set_index('run_name').loc[top3, 'oof_auc'].values
    auc_weights = np.maximum(aucs - aucs.min() + 1e-5, 1e-5)
    add_blend_row(rank_rows, top3, auc_weights, kind='rank')

rank_blend_df = (
    pd.DataFrame(rank_rows)
    .sort_values(['oof_auc', 'oof_logloss'], ascending=[False, True])
    .reset_index(drop=True)
    if rank_rows else pd.DataFrame()
)
display(rank_blend_df.head(30) if not rank_blend_df.empty else rank_blend_df)


#### 03-9. 결과 해석

Rank Average는 예측 확률의 절대값 대신 순위를 평균냅니다.  
ROC-AUC에는 유리할 수 있지만, 확률 보정이 약해져 LogLoss는 나빠질 수 있습니다.


### 03-10. Raw + Rank Hybrid Ensemble


In [ ]:
# Raw + Rank Hybrid 앙상블

all_blends = [raw_blend_df]
if not fine_blend_df.empty:
    all_blends.append(fine_blend_df)
if not rank_blend_df.empty:
    all_blends.append(rank_blend_df)

blend_summary = (
    pd.concat(all_blends, ignore_index=True)
    .drop_duplicates(['kind', 'members', 'weights'])
    .sort_values(['oof_auc', 'oof_logloss'], ascending=[False, True])
    .reset_index(drop=True)
)

hybrid_rows = []
if not raw_blend_df.empty and not rank_blend_df.empty:
    best_raw = raw_blend_df.iloc[0]
    best_rank = rank_blend_df.iloc[0]
    raw_names = best_raw['members'].split('|')
    raw_weights = np.array([float(x) for x in best_raw['weights'].split('|')])
    rank_names = best_rank['members'].split('|')
    rank_weights = np.array([float(x) for x in best_rank['weights'].split('|')])

    raw_oof, raw_test = weighted_average(raw_names, raw_weights, use_rank=False)
    rank_oof, rank_test = weighted_average(rank_names, rank_weights, use_rank=True)

    for alpha in np.linspace(0.1, 0.9, 17):
        oof_h = clip_pred(alpha * raw_oof + (1 - alpha) * rank_oof)
        test_h = clip_pred(alpha * raw_test + (1 - alpha) * rank_test)
        hybrid_rows.append({
            'blend_name': 'hybrid_raw_rank',
            'kind': 'hybrid',
            'members': f"RAW[{best_raw['members']}] + RANK[{best_rank['members']}]",
            'weights': f'raw_alpha={alpha:.2f}',
            'oof_auc': roc_auc_score(y, oof_h),
            'oof_logloss': log_loss(y, oof_h),
            '_oof': oof_h,
            '_test_pred': test_h,
        })

hybrid_df = pd.DataFrame(hybrid_rows).sort_values(['oof_auc', 'oof_logloss'], ascending=[False, True]).reset_index(drop=True) if hybrid_rows else pd.DataFrame()

blend_summary_public = blend_summary.copy()
if not hybrid_df.empty:
    hybrid_public = hybrid_df.drop(columns=['_oof', '_test_pred'])
    blend_summary_public = pd.concat([blend_summary_public, hybrid_public], ignore_index=True)
    blend_summary_public = blend_summary_public.sort_values(['oof_auc', 'oof_logloss'], ascending=[False, True]).reset_index(drop=True)

display(blend_summary_public.head(40))
blend_summary_public.to_csv(cfg.out_dir / 'blend_scores.csv', index=False)


#### 03-10. 결과 해석

Hybrid는 raw 평균과 rank 평균을 다시 섞는 방식입니다.  
다만 OOF 차이가 매우 작으면 단일 최고 모델, raw 최고, rank 최고, hybrid 최고를 모두 보관하고 실제 제출 후보는 신중히 선택합니다.


## 04. 평가


### 04-1. 평가 지표 정리


In [ ]:
# 모델별/앙상블별 최종 성능 비교

final_compare = []
for _, row in score_df.iterrows():
    final_compare.append({
        'type': 'model',
        'name': row['run_name'],
        'kind': row['model_name'],
        'members': row['run_name'],
        'weights': '1.0',
        'oof_auc': row['oof_auc'],
        'oof_logloss': row['oof_logloss'],
    })

for _, row in blend_summary_public.head(20).iterrows():
    final_compare.append({
        'type': 'blend',
        'name': row['blend_name'],
        'kind': row['kind'],
        'members': row['members'],
        'weights': row['weights'],
        'oof_auc': row['oof_auc'],
        'oof_logloss': row['oof_logloss'],
    })

final_compare_df = pd.DataFrame(final_compare).sort_values(['oof_auc', 'oof_logloss'], ascending=[False, True]).reset_index(drop=True)
display(final_compare_df.head(30))
final_compare_df.to_csv(cfg.out_dir / 'final_compare.csv', index=False)


### 04-2. 최종 후보 선택 기준

최종 후보는 `final_compare_df`의 1위입니다.  
단, 1위가 rank 또는 hybrid인 경우 확률값 분포가 왜곡될 수 있으므로, raw 최고 후보와 단일 모델 최고 후보도 함께 저장합니다.


In [ ]:
# 최종 후보 예측값 생성 함수


def resolve_candidate(row: pd.Series) -> Tuple[np.ndarray, np.ndarray, str]:
    kind = row['kind']
    if kind in ['raw', 'rank']:
        names = row['members'].split('|')
        weights = np.array([float(x) for x in row['weights'].split('|')])
        oof_pred, test_pred = weighted_average(names, weights, use_rank=(kind == 'rank'))
        label = f"{kind}_{'_'.join(names)}"
        return oof_pred, test_pred, label
    if kind == 'hybrid' and not hybrid_df.empty:
        idx = hybrid_df.index[
            (hybrid_df['members'] == row['members']) &
            (hybrid_df['weights'] == row['weights'])
        ]
        if len(idx):
            rec = hybrid_df.loc[idx[0]]
            return rec['_oof'], rec['_test_pred'], 'hybrid_raw_rank'
    # fallback: single model row from final_compare
    name = str(row['members']).split('|')[0]
    return get_oof(name), get_test_pred(name), name

selected = final_compare_df.iloc[0]
selected_oof, selected_test, selected_label = resolve_candidate(selected)

print('selected label:', selected_label)
print('selected kind:', selected['kind'])
print('selected members:', selected['members'])
print('selected weights:', selected['weights'])
print('selected OOF ROC-AUC:', roc_auc_score(y, selected_oof))
print('selected OOF LogLoss:', log_loss(y, selected_oof))


## 05. 제출 및 리더보드 확인


### 05-1. 제출 파일 생성


In [ ]:
# 최종 제출 후보 파일 생성

pred_cols = [col for col in sample_submission.columns if col != cfg.id_col]
if len(pred_cols) != 1:
    raise ValueError('sample_submission의 예측 컬럼을 하나로 특정할 수 없습니다.')
pred_col = pred_cols[0]

submission = sample_submission.copy()
submission[pred_col] = clip_pred(selected_test)

safe_label = re.sub(r'[^A-Za-z0-9가-힣_\-]+', '_', selected_label)[:120]
submission_path = cfg.out_dir / f'submission_team_report_v6_1_{safe_label}.csv'
oof_path = cfg.out_dir / f'oof_team_report_v6_1_{safe_label}.csv'

submission.to_csv(submission_path, index=False)
pd.DataFrame({
    cfg.id_col: train[cfg.id_col],
    cfg.target: y,
    'prediction': selected_oof,
}).to_csv(oof_path, index=False)

print('saved submission:', submission_path)
print('saved oof:', oof_path)
display(submission.head())


### 05-2. 제출 파일 검증


In [ ]:
# 제출 파일 검증


def validate_submission(submission: pd.DataFrame, sample_submission: pd.DataFrame, cfg: Config) -> None:
    pred_cols = [col for col in sample_submission.columns if col != cfg.id_col]
    if len(pred_cols) != 1:
        raise ValueError('sample_submission의 예측 컬럼을 하나로 특정할 수 없습니다.')
    pred_col = pred_cols[0]
    checks = {
        'same_shape': submission.shape == sample_submission.shape,
        'same_columns': submission.columns.tolist() == sample_submission.columns.tolist(),
        'same_id_order': submission[cfg.id_col].astype(str).tolist() == sample_submission[cfg.id_col].astype(str).tolist(),
        'no_nan': int(submission[pred_col].isna().sum()) == 0,
        'between_0_1': bool(submission[pred_col].between(0, 1).all()),
    }
    display(pd.Series(checks, name='check'))
    failed = [name for name, ok in checks.items() if not ok]
    if failed:
        raise ValueError('제출 파일 검증 실패: ' + ', '.join(failed))
    print('제출 파일 검증 완료')
    display(submission[pred_col].describe().to_frame('prediction'))

validate_submission(submission, sample_submission, cfg)


### 05-3. 실행 기록 저장


In [ ]:
# 실행 기록 저장

run_manifest = {
    'run_mode': cfg.run_mode,
    'n_splits': cfg.n_splits,
    'seeds': cfg.seeds(),
    'target': cfg.target,
    'id_col': cfg.id_col,
    'data_dir': str(cfg.data_dir),
    'out_dir': str(cfg.out_dir),
    'feature_count': len(feature_cols),
    'categorical_count': len(cat_cols),
    'numeric_count': len(num_cols),
    'selected': selected.to_dict(),
    'selected_label': selected_label,
    'selected_oof_auc': float(roc_auc_score(y, selected_oof)),
    'selected_oof_logloss': float(log_loss(y, selected_oof)),
    'submission_path': str(submission_path),
    'oof_path': str(oof_path),
}

with open(cfg.out_dir / 'run_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(run_manifest, f, ensure_ascii=False, indent=2)

summary_md = f"""
# Team Report v6.1 실행 요약

- run_mode: {cfg.run_mode}
- feature_count: {len(feature_cols)}
- categorical_count: {len(cat_cols)}
- numeric_count: {len(num_cols)}
- selected_label: {selected_label}
- selected_kind: {selected['kind']}
- selected_members: {selected['members']}
- selected_weights: {selected['weights']}
- selected_oof_auc: {roc_auc_score(y, selected_oof):.8f}
- selected_oof_logloss: {log_loss(y, selected_oof):.8f}
- submission: `{submission_path}`
- oof: `{oof_path}`
""".strip()

with open(cfg.out_dir / 'run_summary.md', 'w', encoding='utf-8') as f:
    f.write(summary_md + '\n')

print(summary_md)


## 최종 정리

v6.1은 v5와 비슷하게 읽히도록 EDA → 전처리 → 모델링 → 앙상블 → 제출 순서로 세분화했습니다.  
다만 v5 후반부에서처럼 특정 셀을 먼저 실행해야만 존재하는 변수에 의존하지 않도록, 모든 모델 결과는 `prediction_bank`, 모든 점수는 `score_df`, 모든 앙상블 후보는 `blend_summary_public`으로 관리합니다.

실행 권장 순서는 다음과 같습니다.

1. `cfg = Config(run_mode='quick')` 상태로 전체 실행합니다.  
2. 끝까지 통과하면 `cfg = Config(run_mode='full')`로 바꾸어 다시 실행합니다.  
3. `outputs/team_report_v6_1/`에 저장된 `submission_*.csv`, `oof_*.csv`, `run_summary.md`를 확인합니다.
